# LTX 200M Student Distillation (Teacher-Student)

Цель ноутбука:
- взять `Lightricks/LTX-Video` как teacher (модель из `ltx_benchmark.py`),
- автоматически собрать student-трансформер около **200M параметров**,
- обучить student на ваших портретных видео через distillation,
- сохранить student как отдельный checkpoint.

> Важно: API `diffusers` для LTX может немного отличаться между версиями. В ноутбуке есть `inspect.signature(...)` и безопасные helper-функции, чтобы адаптация была минимальной.

In [ ]:
import os
import math
import json
import inspect
import random
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from diffusers import LTXImageToVideoPipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

In [ ]:
@dataclass
class DistillConfig:
    # Teacher from ltx_benchmark.py
    teacher_model_id: str = "Lightricks/LTX-Video"
    dtype: torch.dtype = torch.bfloat16

    # Data
    dataset_root: str = "openvid_portrait_subset"
    manifest_name: str = "subset_manifest.csv"
    max_train_samples: Optional[int] = None

    # Video preprocessing (match your LTX benchmark defaults)
    frames: int = 25
    width: int = 512
    height: int = 512

    # Student target
    target_params_millions: float = 200.0

    # I2V conditioning
    conditioning_strength: float = 1.0

    # Training
    batch_size: int = 1
    grad_accum_steps: int = 2
    num_workers: int = 2
    epochs: int = 5
    lr: float = 1e-4
    weight_decay: float = 1e-2
    max_grad_norm: float = 1.0

    # Distillation weights
    loss_mse_weight: float = 1.0
    loss_cos_weight: float = 0.05

    # IO
    output_dir: str = "outputs_ltx_student_200m"
    save_every_steps: int = 250
    save_every_videos: int = 5000

cfg = DistillConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(asdict(cfg))

In [ ]:
def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


def _find_first_config_key(config: Dict, keys: List[str]) -> Optional[str]:
    for k in keys:
        if k in config:
            return k
    return None


def make_student_from_teacher(teacher_transformer: nn.Module, target_m: float = 200.0) -> nn.Module:
    """
    Создает student из teacher.config, уменьшая ТОЛЬКО глубину (число блоков).

    Это сохраняет входной/выходной интерфейс (proj_in/proj_out), что критично для LTX,
    и резко снижает риск shape-mismatch в трансформере.
    """
    teacher_cls = teacher_transformer.__class__
    teacher_cfg = dict(teacher_transformer.config)

    layers_key = _find_first_config_key(teacher_cfg, ["num_layers", "num_hidden_layers", "layers_per_block", "depth"])
    if layers_key is None or not isinstance(teacher_cfg.get(layers_key), int):
        raise RuntimeError(f"Не удалось найти integer depth key в config: {teacher_cfg.keys()}")

    teacher_params = count_params_m(teacher_transformer)
    teacher_layers = int(teacher_cfg[layers_key])
    print(f"Teacher params: {teacher_params:.1f}M, layers={teacher_layers}")

    # Initial ratio estimate assuming params roughly linear in depth.
    ratio = max(0.05, min(1.0, target_m / max(teacher_params, 1e-6)))
    base_layers = max(2, int(round(teacher_layers * ratio)))

    # Try a small neighborhood around estimated depth and pick closest by params.
    candidates = sorted(set([max(2, base_layers + d) for d in (-2, -1, 0, 1, 2)]))

    best_model = None
    best_gap = float("inf")

    for n_layers in candidates:
        c = dict(teacher_cfg)
        c[layers_key] = int(n_layers)
        m = teacher_cls.from_config(c)
        p = count_params_m(m)
        gap = abs(p - target_m)
        print(f"  candidate layers={n_layers}: {p:.1f}M")
        if gap < best_gap:
            best_gap = gap
            best_model = m

    assert best_model is not None
    print(f"Student params: {count_params_m(best_model):.1f}M (target={target_m:.1f}M)")
    return best_model


teacher_pipe = LTXImageToVideoPipeline.from_pretrained(
    cfg.teacher_model_id,
    torch_dtype=cfg.dtype,
).to(device)
teacher_pipe.set_progress_bar_config(disable=True)

teacher_transformer = teacher_pipe.transformer.eval()
for p in teacher_transformer.parameters():
    p.requires_grad_(False)

student_transformer = make_student_from_teacher(teacher_transformer, cfg.target_params_millions).to(device)
print("Teacher class:", teacher_transformer.__class__.__name__)
print("Student class:", student_transformer.__class__.__name__)
print("Teacher in_channels:", getattr(teacher_transformer.config, "in_channels", None))

In [ ]:
import cv2


def read_video_cv2(path: str, num_frames: int, width: int, height: int) -> torch.Tensor:
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        raise RuntimeError(f"empty/broken video: {path}")

    # uniform sampling to fixed frame count
    idx = np.linspace(0, len(frames) - 1, num_frames).round().astype(int)
    sampled = [frames[i] for i in idx]
    arr = np.stack(sampled).astype(np.float32) / 255.0  # [F,H,W,C]
    tensor = torch.from_numpy(arr).permute(0, 3, 1, 2)  # [F,C,H,W]
    tensor = tensor * 2.0 - 1.0
    return tensor


class PortraitVideoDataset(Dataset):
    def __init__(self, cfg: DistillConfig):
        root = Path(cfg.dataset_root)
        manifest = root / cfg.manifest_name
        videos_dir = root / "videos"

        df = pd.read_csv(manifest)
        if cfg.max_train_samples is not None:
            df = df.iloc[: cfg.max_train_samples].copy()

        self.samples = []
        for _, row in df.iterrows():
            video_file = str(row["video_file"])
            caption = str(row.get("caption", "portrait animation"))
            path = videos_dir / video_file
            if path.exists():
                self.samples.append((str(path), caption))

        self.frames = cfg.frames
        self.width = cfg.width
        self.height = cfg.height

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, caption = self.samples[idx]
        pix = read_video_cv2(path, self.frames, self.width, self.height)
        cond_image = pix[0].clone()  # first frame as I2V conditioning image
        return {
            "pixel_values": pix,
            "conditioning_image": cond_image,
            "caption": caption,
            "path": path,
        }


ds = PortraitVideoDataset(cfg)
print("dataset size:", len(ds))

loader = DataLoader(
    ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
)

batch = next(iter(loader))
print(batch["pixel_values"].shape, len(batch["caption"]))

In [ ]:
def encode_prompt_batch(pipe, captions: List[str], dtype: torch.dtype):
    with torch.no_grad():
        prompt_embeds, prompt_attention_mask, _, _ = pipe.encode_prompt(
            prompt=captions,
            negative_prompt=["" for _ in captions],
            do_classifier_free_guidance=False,
            num_videos_per_prompt=1,
            max_sequence_length=128,
            device=device,
            dtype=dtype,
        )
    return prompt_embeds, prompt_attention_mask


def encode_video_to_latents(pipe, pixel_values: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
    """pixel_values: [B,F,C,H,W] -> latents: [B,C,F,H',W']"""
    b, f, c, h, w = pixel_values.shape
    x = pixel_values.permute(0, 2, 1, 3, 4).contiguous().to(device=device, dtype=dtype)

    with torch.no_grad():
        encoded = pipe.vae.encode(x)
        z = encoded.latent_dist.sample() if hasattr(encoded, "latent_dist") else encoded[0]

    if hasattr(pipe.vae, "config") and hasattr(pipe.vae.config, "scaling_factor"):
        z = z * pipe.vae.config.scaling_factor
    return z


def add_noise_flowmatch(scheduler, latents: torch.Tensor, noise: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
    # Target environment: FlowMatchEulerDiscreteScheduler
    return scheduler.scale_noise(sample=latents, timestep=timesteps, noise=noise)


def encode_conditioning_image_to_latent(pipe, conditioning_image: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
    """conditioning_image: [B,C,H,W] in [-1,1] -> [B,C_lat,H',W']"""
    x = conditioning_image.to(device=device, dtype=dtype)
    x5d = x.unsqueeze(2)  # [B,C,1,H,W] expected by this LTX VAE

    with torch.no_grad():
        encoded = pipe.vae.encode(x5d)
        z = encoded.latent_dist.sample() if hasattr(encoded, "latent_dist") else encoded[0]

    if hasattr(pipe.vae, "config") and hasattr(pipe.vae.config, "scaling_factor"):
        z = z * pipe.vae.config.scaling_factor

    # [B,C,1,H',W'] -> [B,C,H',W'] for frame conditioning
    if z.ndim == 5:
        z = z[:, :, 0, :, :]
    return z


def apply_i2v_conditioning(noisy_latents: torch.Tensor, image_latents: torch.Tensor, strength: float = 1.0) -> torch.Tensor:
    """
    Inject image conditioning into the first latent frame.
    noisy_latents: [B,C,F,H,W]
    image_latents: [B,C,H,W]
    """
    out = noisy_latents.clone()
    target = out[:, :, 0, :, :]

    img = image_latents
    if img.shape[-2:] != target.shape[-2:]:
        img = F.interpolate(img, size=target.shape[-2:], mode="bilinear", align_corners=False)

    if img.shape[1] != target.shape[1]:
        if target.shape[1] % img.shape[1] == 0:
            img = img.repeat(1, target.shape[1] // img.shape[1], 1, 1)
        else:
            raise RuntimeError(f"I2V channel mismatch: image={img.shape[1]}, target={target.shape[1]}")

    out[:, :, 0, :, :] = (1.0 - strength) * target + strength * img
    return out


def _align_token_grid(payload: Dict) -> Dict:
    out = dict(payload)
    hs = out.get("hidden_states")
    if not isinstance(hs, torch.Tensor) or hs.ndim != 3:
        return out

    n_tokens = int(hs.shape[1])
    nf = int(out.get("num_frames", 1))
    area = max(1, n_tokens // max(1, nf)) if n_tokens % max(1, nf) == 0 else n_tokens
    side = int(math.sqrt(area))
    if side * side == area:
        out["height"] = side
        out["width"] = side
        out["num_frames"] = nf if n_tokens % max(1, nf) == 0 else 1
    else:
        out["num_frames"] = 1
        out["height"] = 1
        out["width"] = n_tokens
    return out


def _call_method_flexible(obj, method_name: str, **kwargs):
    fn = getattr(obj, method_name)
    sig = inspect.signature(fn)
    allowed = set(sig.parameters.keys())
    payload = {k: v for k, v in kwargs.items() if k in allowed}
    return fn(**payload)


def try_pack_hidden_states(pipe, latents_5d: torch.Tensor) -> Optional[torch.Tensor]:
    """Try native LTX latent packing helpers to produce transformer-ready hidden states."""
    if latents_5d.ndim != 5:
        return None
    b, _c, f, h, w = latents_5d.shape

    for method_name in ["_pack_latents", "pack_latents"]:
        if not hasattr(pipe, method_name):
            continue
        try:
            packed = _call_method_flexible(
                pipe,
                method_name,
                latents=latents_5d,
                batch_size=b,
                num_frames=f,
                height=h,
                width=w,
            )
            if isinstance(packed, (tuple, list)):
                packed = packed[0]
            if isinstance(packed, torch.Tensor) and packed.ndim == 3:
                print(f"[pack] using {method_name}: {tuple(latents_5d.shape)} -> {tuple(packed.shape)}")
                return packed
        except Exception:
            pass
    return None


def _adapt_hidden_states_fallback(payload: Dict) -> Dict:
    """Fallback only if native packing is unavailable in this diffusers build."""
    out = dict(payload)
    hs = out.get("hidden_states")
    if not isinstance(hs, torch.Tensor):
        return out

    original_shape = tuple(hs.shape)

    if hs.ndim == 5 and hs.shape[1] == 16:
        b, c, f, h, w = hs.shape
        hs = hs.permute(0, 2, 3, 4, 1).reshape(b, f * h * w, c)
    elif hs.ndim == 4 and hs.shape[1] == 16:
        b, c, h, w = hs.shape
        hs = hs.permute(0, 2, 3, 1).reshape(b, h * w, c)
    elif hs.ndim == 2 and hs.shape[-1] == 16:
        hs = hs.unsqueeze(0)

    if hs.ndim >= 2 and hs.shape[-1] == 16:
        hs = hs.repeat(*([1] * (hs.ndim - 1)), 8)

    print(f"[fallback] hidden_states: {original_shape} -> {tuple(hs.shape)}")
    out["hidden_states"] = hs
    if "sample" in out and isinstance(out["sample"], torch.Tensor):
        out["sample"] = hs
    return _align_token_grid(out)


def prepare_hidden_states_for_transformer(pipe, noisy_latents: torch.Tensor) -> torch.Tensor:
    packed = try_pack_hidden_states(pipe, noisy_latents)
    if packed is not None:
        return packed

    # Fallback to previous compatibility path.
    payload = {"hidden_states": noisy_latents, "sample": noisy_latents}
    payload = _adapt_hidden_states_fallback(payload)
    return payload["hidden_states"]


def call_transformer_flexible(model: nn.Module, **kwargs):
    sig = inspect.signature(model.forward)
    allowed = set(sig.parameters.keys())
    payload = {k: v for k, v in kwargs.items() if k in allowed}
    payload = _align_token_grid(payload)

    out = model(**payload)

    if hasattr(out, "sample"):
        return out.sample
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


print("Teacher transformer forward signature:")
print(inspect.signature(teacher_transformer.forward))

In [ ]:
optimizer = torch.optim.AdamW(
    student_transformer.parameters(),
    lr=cfg.lr,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)

scheduler = teacher_pipe.scheduler
num_ts = int(getattr(scheduler.config, "num_train_timesteps", 1000))

# Fixed path for current environment: FlowMatch + dynamic shifting.
scheduler.set_timesteps(num_ts, device=device, mu=1.0)
print("scheduler:", scheduler.__class__.__name__)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

student_transformer.train()
global_step = 0
videos_seen = 0
next_video_ckpt_at = cfg.save_every_videos
start_epoch = 0


def save_intermediate_checkpoint(epoch: int, step: int) -> None:
    """
    Keep exactly ONE checkpoint directory and overwrite atomically.
    """
    root = Path(cfg.output_dir)
    target_dir = root / "checkpoint-latest"
    tmp_dir = root / "checkpoint-latest.tmp"
    bak_dir = root / "checkpoint-latest.bak"

    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # 1) Save everything into temp dir first.
    student_transformer.save_pretrained(tmp_dir / "student_transformer")

    train_state = {
        "epoch": epoch,
        "step_in_epoch": step,
        "global_step": global_step,
        "videos_seen": videos_seen,
        "next_video_ckpt_at": next_video_ckpt_at,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "config": asdict(cfg),
    }
    torch.save(train_state, tmp_dir / "train_state.pt")

    with (tmp_dir / "distill_config.json").open("w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, ensure_ascii=False, indent=2, default=str)

    # 2) Atomic-ish swap: target -> bak, tmp -> target.
    if bak_dir.exists():
        shutil.rmtree(bak_dir)
    if target_dir.exists():
        target_dir.rename(bak_dir)
    tmp_dir.rename(target_dir)

    # 3) Cleanup backup if new checkpoint is in place.
    if bak_dir.exists():
        shutil.rmtree(bak_dir)


def find_latest_checkpoint_dir(output_dir: str) -> Optional[Path]:
    root = Path(output_dir)
    if not root.exists():
        return None

    latest = root / "checkpoint-latest"
    if latest.exists() and (latest / "train_state.pt").exists() and (latest / "student_transformer").exists():
        return latest

    # Backward compatibility with older multi-checkpoint runs.
    candidates = sorted(
        [p for p in root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for ckpt in candidates:
        if (ckpt / "train_state.pt").exists() and (ckpt / "student_transformer").exists():
            return ckpt
    return None


def auto_resume_if_available() -> None:
    global global_step, videos_seen, next_video_ckpt_at, start_epoch

    ckpt_dir = find_latest_checkpoint_dir(cfg.output_dir)
    if ckpt_dir is None:
        print("No resume checkpoint found. Starting from scratch.")
        return

    print(f"Resuming from: {ckpt_dir}")

    # Restore model weights into the already-created student instance.
    restored = student_transformer.__class__.from_pretrained(ckpt_dir / "student_transformer")
    student_transformer.load_state_dict(restored.state_dict())
    student_transformer.to(device)
    del restored

    state = torch.load(ckpt_dir / "train_state.pt", map_location="cpu")

    if "optimizer" in state:
        optimizer.load_state_dict(state["optimizer"])
        # Move optimizer state tensors back to active device.
        for opt_state in optimizer.state.values():
            for k, v in list(opt_state.items()):
                if torch.is_tensor(v):
                    opt_state[k] = v.to(device)
    if "scaler" in state:
        scaler.load_state_dict(state["scaler"])

    global_step = int(state.get("global_step", 0))
    videos_seen = int(state.get("videos_seen", 0))
    next_video_ckpt_at = int(state.get("next_video_ckpt_at", cfg.save_every_videos))

    # Continue from the same epoch index; dataloader order may differ after restart, but training resumes from latest weights.
    start_epoch = int(state.get("epoch", 0))

    # Safety: keep checkpoint threshold ahead of current progress.
    while next_video_ckpt_at <= videos_seen:
        next_video_ckpt_at += cfg.save_every_videos

    print(
        f"Resume state: start_epoch={start_epoch}, global_step={global_step}, "
        f"videos_seen={videos_seen}, next_video_ckpt_at={next_video_ckpt_at}"
    )


auto_resume_if_available()

for epoch in range(start_epoch, cfg.epochs):
    for step, batch in enumerate(loader):
        pixel_values = batch["pixel_values"]  # [B,F,C,H,W]
        captions = list(batch["caption"])
        conditioning_image = batch["conditioning_image"]  # [B,C,H,W]

        bsz = pixel_values.shape[0]
        videos_seen += bsz

        with torch.no_grad():
            latents = encode_video_to_latents(teacher_pipe, pixel_values, cfg.dtype)
            image_latents = encode_conditioning_image_to_latent(teacher_pipe, conditioning_image, cfg.dtype)
            prompt_embeds, prompt_attn = encode_prompt_batch(teacher_pipe, captions, cfg.dtype)

            noise = torch.randn_like(latents)
            idx = torch.randint(0, scheduler.timesteps.numel(), (bsz,), device=device)
            timesteps = scheduler.timesteps.to(device=device)[idx]
            noisy_latents = add_noise_flowmatch(scheduler, latents, noise, timesteps)
            noisy_latents = apply_i2v_conditioning(noisy_latents, image_latents, strength=cfg.conditioning_strength)

            latent_num_frames = noisy_latents.shape[2]
            latent_h = noisy_latents.shape[3]
            latent_w = noisy_latents.shape[4]

            hidden_states = prepare_hidden_states_for_transformer(teacher_pipe, noisy_latents)

            teacher_pred = call_transformer_flexible(
                teacher_transformer,
                hidden_states=hidden_states,
                sample=hidden_states,
                timestep=timesteps,
                timesteps=timesteps,
                encoder_hidden_states=prompt_embeds,
                encoder_attention_mask=prompt_attn,
                num_frames=latent_num_frames,
                height=latent_h,
                width=latent_w,
                return_dict=False,
            )

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda"), dtype=torch.bfloat16):
            student_pred = call_transformer_flexible(
                student_transformer,
                hidden_states=hidden_states,
                sample=hidden_states,
                timestep=timesteps,
                timesteps=timesteps,
                encoder_hidden_states=prompt_embeds,
                encoder_attention_mask=prompt_attn,
                num_frames=latent_num_frames,
                height=latent_h,
                width=latent_w,
                return_dict=False,
            )

            loss_mse = F.mse_loss(student_pred.float(), teacher_pred.float())
            loss_cos = 1.0 - F.cosine_similarity(
                student_pred.float().flatten(1),
                teacher_pred.float().flatten(1),
                dim=1,
            ).mean()
            loss = cfg.loss_mse_weight * loss_mse + cfg.loss_cos_weight * loss_cos
            loss = loss / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student_transformer.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            while videos_seen >= next_video_ckpt_at:
                save_intermediate_checkpoint(epoch=epoch, step=step)
                print(f"saved video checkpoint at videos_seen={videos_seen} (target={next_video_ckpt_at})")
                next_video_ckpt_at += cfg.save_every_videos

            if global_step % 10 == 0:
                print(
                    f"epoch={epoch} step={global_step} "
                    f"loss={loss.item() * cfg.grad_accum_steps:.5f} "
                    f"mse={loss_mse.item():.5f} cos={loss_cos.item():.5f}"
                )

            if global_step % cfg.save_every_steps == 0:
                save_intermediate_checkpoint(epoch=epoch, step=step)

print("Training finished. global_step=", global_step, "videos_seen=", videos_seen)

In [ ]:
final_dir = Path(cfg.output_dir) / "final"
final_dir.mkdir(parents=True, exist_ok=True)

student_transformer.eval()
student_transformer.save_pretrained(final_dir / "student_transformer")

with (final_dir / "distill_config.json").open("w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2, default=str)

# Save a full pipeline folder with student transformer swapped in
student_pipe = LTXImageToVideoPipeline.from_pretrained(
    cfg.teacher_model_id,
    torch_dtype=cfg.dtype,
)
student_pipe.transformer = student_transformer.cpu()
student_pipe.save_pretrained(final_dir / "student_pipeline")

print("Saved:")
print(" -", final_dir / "student_transformer")
print(" -", final_dir / "student_pipeline")

## Если словили API mismatch

Если в вашей версии `diffusers` сигнатура `teacher_transformer.forward(...)` отличается:

1. Выполните ячейку с `inspect.signature(teacher_transformer.forward)`.
2. В `call_transformer_flexible(...)` уже есть фильтрация unsupported kwargs.
3. Обычно достаточно переименовать один из аргументов в train-loop:
   - `hidden_states`
   - `timestep` / `timesteps`
   - `encoder_hidden_states`
   - `encoder_attention_mask`

## Рекомендуемый старт обучения

- сначала `max_train_samples=2000`, `epochs=1` (smoke run),
- затем полный прогон на всем вашем портретном подмножестве,
- после этого сравнить teacher vs student в том же пайплайне, что в `ltx_benchmark.py` (same prompt/image/seed).